# Final Model Selection

This notebook performs the final validation and selection of the customer segmentation model.

The objective is to confirm the best clustering configuration, evaluate its stability and business usefulness, and prepare the final preprocessing and clustering pipeline for production.

In [1]:
import pandas as pd
import numpy as np

file_path = "../data/processed/customer_features_raw.csv"

customer_df = pd.read_csv(file_path)

print("Shape:", customer_df.shape)

print("\nColumns:")
print(customer_df.columns.tolist())

print("\nMissing values:")
print(customer_df.isna().sum())

display(customer_df.head())

Shape: (5350, 7)

Columns:
['Customer ID', 'Recency', 'Frequency', 'Monetary', 'Total_Quantity', 'Unique_Products', 'Average_Order_Value']

Missing values:
Customer ID            0
Recency                0
Frequency              0
Monetary               0
Total_Quantity         0
Unique_Products        0
Average_Order_Value    0
dtype: int64


,Customer ID,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
0,12346.0,326,12,77556.46,74285,27,6463.038333
1,12608.0,405,1,415.79,323,16,415.790000
2,12745.0,487,2,723.85,467,20,361.925000
3,12746.0,541,1,254.55,97,17,254.550000
4,12747.0,2,26,8898.48,2640,85,342.249231


In [2]:
features = [
    "Recency",
    "Frequency",
    "Monetary",
    "Total_Quantity",
    "Unique_Products",
    "Average_Order_Value"
]

X = customer_df[features].copy()

print("===== CLUSTERING FEATURES =====")
print("Shape:", X.shape)

print("\nFeatures:")
print(X.columns.tolist())

display(X.head())

===== CLUSTERING FEATURES =====
Shape: (5350, 6)

Features:
['Recency', 'Frequency', 'Monetary', 'Total_Quantity', 'Unique_Products', 'Average_Order_Value']


,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
0,326,12,77556.46,74285,27,6463.038333
1,405,1,415.79,323,16,415.790000
2,487,2,723.85,467,20,361.925000
3,541,1,254.55,97,17,254.550000
4,2,26,8898.48,2640,85,342.249231


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_holdout = train_test_split(
    X,
    test_size=0.20,
    random_state=42
)

print("===== TRAIN / HOLDOUT SPLIT =====")
print("Total customers:", len(X))
print("Training customers:", len(X_train))
print("Holdout customers:", len(X_holdout))

print("\nTraining proportion:", round(len(X_train) / len(X) * 100, 2), "%")
print("Holdout proportion:", round(len(X_holdout) / len(X) * 100, 2), "%")

===== TRAIN / HOLDOUT SPLIT =====
Total customers: 5350
Training customers: 4280
Holdout customers: 1070

Training proportion: 80.0 %
Holdout proportion: 20.0 %


In [4]:
from sklearn.preprocessing import StandardScaler

log_features = [
    "Frequency",
    "Monetary",
    "Total_Quantity",
    "Unique_Products",
    "Average_Order_Value"
]

# Copy the datasets
X_train_processed = X_train.copy()
X_holdout_processed = X_holdout.copy()

# Log transformation
for col in log_features:
    X_train_processed[col] = np.log1p(X_train_processed[col])
    X_holdout_processed[col] = np.log1p(X_holdout_processed[col])

# Fit scaler ONLY on training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_processed)

# Transform holdout using the training-fitted scaler
X_holdout_scaled = scaler.transform(X_holdout_processed)

print("===== PREPROCESSING COMPLETE =====")
print("Training shape:", X_train_scaled.shape)
print("Holdout shape:", X_holdout_scaled.shape)

print("\nTraining means:")
print(np.round(X_train_scaled.mean(axis=0), 4))

print("\nTraining standard deviations:")
print(np.round(X_train_scaled.std(axis=0), 4))

===== PREPROCESSING COMPLETE =====
Training shape: (4280, 6)
Holdout shape: (1070, 6)

Training means:
[-0.  0. -0. -0. -0. -0.]

Training standard deviations:
[1. 1. 1. 1. 1. 1.]


In [5]:
from sklearn.cluster import KMeans

results = []

for k in [2, 3, 4]:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    train_labels = model.fit_predict(X_train_scaled)
    holdout_labels = model.predict(X_holdout_scaled)

    results.append({
        "K": k,
        "Train_Inertia": model.inertia_,
        "Train_Clusters": len(np.unique(train_labels)),
        "Holdout_Clusters": len(np.unique(holdout_labels))
    })

results_df = pd.DataFrame(results)

print("===== CANDIDATE K-MEANS MODELS =====")
display(results_df)

===== CANDIDATE K-MEANS MODELS =====


,K,Train_Inertia,Train_Clusters,Holdout_Clusters
0,2,14333.470636,2,2
1,3,11155.042777,3,3
2,4,9689.644610,4,4


In [6]:
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

evaluation_results = []

for k in [2, 3, 4]:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    # Fit only on training data
    model.fit(X_train_scaled)

    # Assign unseen holdout customers
    holdout_labels = model.predict(X_holdout_scaled)

    # Evaluate clustering structure on holdout
    silhouette = silhouette_score(
        X_holdout_scaled,
        holdout_labels
    )

    davies_bouldin = davies_bouldin_score(
        X_holdout_scaled,
        holdout_labels
    )

    calinski_harabasz = calinski_harabasz_score(
        X_holdout_scaled,
        holdout_labels
    )

    evaluation_results.append({
        "K": k,
        "Holdout_Silhouette": silhouette,
        "Holdout_Davies_Bouldin": davies_bouldin,
        "Holdout_Calinski_Harabasz": calinski_harabasz
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("===== HOLDOUT EVALUATION =====")
display(evaluation_df)

===== HOLDOUT EVALUATION =====


,K,Holdout_Silhouette,Holdout_Davies_Bouldin,Holdout_Calinski_Harabasz
0,2,0.34250,1.083230,734.602505
1,3,0.26697,1.223059,593.006769
2,4,0.26492,1.187991,537.429011


In [7]:

from sklearn.metrics import adjusted_rand_score

seeds = [0, 1, 21, 42, 100]

cluster_labels = {}

for seed in seeds:
    model = KMeans(
        n_clusters=2,
        random_state=seed,
        n_init=10
    )

    labels = model.fit_predict(X_train_scaled)
    cluster_labels[seed] = labels

stability_results = []

reference_seed = 42
reference_labels = cluster_labels[reference_seed]

for seed in seeds:
    ari = adjusted_rand_score(
        reference_labels,
        cluster_labels[seed]
    )

    stability_results.append({
        "Seed": seed,
        "ARI_vs_seed_42": ari
    })

stability_df = pd.DataFrame(stability_results)

print("===== K-MEANS STABILITY =====")
display(stability_df)

===== K-MEANS STABILITY =====


,Seed,ARI_vs_seed_42
0,0,0.995331
1,1,0.997198
2,21,1.000000
3,42,1.000000
4,100,1.000000


In [8]:
final_k = 2
final_random_state = 42
final_n_init = 10

print("===== FINAL MODEL SELECTION =====")
print("Algorithm: K-Means")
print("Number of clusters:", final_k)
print("Random state:", final_random_state)
print("n_init:", final_n_init)
print("Selection status: SELECTED")

===== FINAL MODEL SELECTION =====
Algorithm: K-Means
Number of clusters: 2
Random state: 42
n_init: 10
Selection status: SELECTED


## Refit preprocessing + final K-Means on all customers

In [9]:
# Copy the complete feature dataset
X_final = X.copy()

# Apply log transformation
for col in log_features:
    X_final[col] = np.log1p(X_final[col])

# Fit scaler on ALL customers
final_scaler = StandardScaler()

X_final_scaled = final_scaler.fit_transform(X_final)

# Final K-Means model
final_model = KMeans(
    n_clusters=final_k,
    random_state=final_random_state,
    n_init=final_n_init
)

final_labels = final_model.fit_predict(X_final_scaled)

print("===== FINAL MODEL FIT =====")
print("Customers used:", len(X_final))
print("Features:", X_final.shape[1])
print("Clusters:", final_model.n_clusters)

print("\nCluster counts:")
print(pd.Series(final_labels).value_counts().sort_index())

print("\nCluster proportions:")
print(
    pd.Series(final_labels)
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

===== FINAL MODEL FIT =====
Customers used: 5350
Features: 6
Clusters: 2

Cluster counts:
0    2714
1    2636
Name: count, dtype: int64

Cluster proportions:
0    0.5073
1    0.4927
Name: proportion, dtype: float64


In [10]:
customer_final = customer_df.copy()

customer_final["Cluster"] = final_labels

print("===== FINAL CUSTOMER SEGMENTATION =====")
print("Shape:", customer_final.shape)

print("\nCluster distribution:")
print(customer_final["Cluster"].value_counts().sort_index())

print("\nSample:")
display(customer_final.head())

===== FINAL CUSTOMER SEGMENTATION =====
Shape: (5350, 8)

Cluster distribution:
Cluster
0    2714
1    2636
Name: count, dtype: int64

Sample:


,Customer ID,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value,Cluster
0,12346.0,326,12,77556.46,74285,27,6463.038333,1
1,12608.0,405,1,415.79,323,16,415.790000,0
2,12745.0,487,2,723.85,467,20,361.925000,0
3,12746.0,541,1,254.55,97,17,254.550000,0
4,12747.0,2,26,8898.48,2640,85,342.249231,1


In [11]:
cluster_profile = (
    customer_final
    .groupby("Cluster")[features]
    .median()
    .round(2)
)

print("===== FINAL CLUSTER PROFILES =====")
display(cluster_profile)

===== FINAL CLUSTER PROFILES =====


,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
Cluster,,,,,,
0,326.0,1.0,334.22,183.5,20.0,197.68
1,37.0,7.0,2182.70,1305.5,101.5,337.72


In [12]:
business_profile = (
    customer_final
    .groupby("Cluster")
    .agg(
        Customers=("Customer ID", "count"),
        Revenue=("Monetary", "sum"),
        Total_Quantity=("Total_Quantity", "sum"),
        Orders=("Frequency", "sum")
    )
    .reset_index()
)

business_profile["Customer_%"] = (
    business_profile["Customers"]
    / business_profile["Customers"].sum()
    * 100
)

business_profile["Revenue_%"] = (
    business_profile["Revenue"]
    / business_profile["Revenue"].sum()
    * 100
)

business_profile["Revenue_per_Customer"] = (
    business_profile["Revenue"]
    / business_profile["Customers"]
)

business_profile["Orders_per_Customer"] = (
    business_profile["Orders"]
    / business_profile["Customers"]
)

business_profile = business_profile.round(2)

print("===== BUSINESS VALUE BY CLUSTER =====")
display(business_profile)

===== BUSINESS VALUE BY CLUSTER =====


,Cluster,Customers,Revenue,Total_Quantity,Orders,Customer_%,Revenue_%,Revenue_per_Customer,Orders_per_Customer
0,0,2714,1100832.27,620128,5044,50.73,7.65,405.61,1.86
1,1,2636,13288402.65,7912283,28497,49.27,92.35,5041.12,10.81


In [13]:
overall_median = customer_final[features].median()

relative_profile = cluster_profile / overall_median

print("===== RELATIVE CLUSTER PROFILE =====")
display(relative_profile.round(2))

===== RELATIVE CLUSTER PROFILE =====


,Recency,Frequency,Monetary,Total_Quantity,Unique_Products,Average_Order_Value
Cluster,,,,,,
0,3.31,0.33,0.40,0.40,0.45,0.73
1,0.38,2.33,2.63,2.81,2.31,1.25


In [14]:
final_silhouette = silhouette_score(
    X_final_scaled,
    final_labels
)

final_davies_bouldin = davies_bouldin_score(
    X_final_scaled,
    final_labels
)

final_calinski_harabasz = calinski_harabasz_score(
    X_final_scaled,
    final_labels
)

print("===== FINAL MODEL METRICS =====")
print("Silhouette Score:", round(final_silhouette, 4))
print("Davies-Bouldin Index:", round(final_davies_bouldin, 4))
print("Calinski-Harabasz Score:", round(final_calinski_harabasz, 2))

===== FINAL MODEL METRICS =====
Silhouette Score: 0.3632
Davies-Bouldin Index: 1.026
Calinski-Harabasz Score: 4151.84


In [15]:
output_path = "../data/processed/final_customer_segmentation.csv"

customer_final.to_csv(
    output_path,
    index=False
)

print("===== FINAL SEGMENTATION SAVED =====")
print("Path:", output_path)
print("Rows:", len(customer_final))
print("Columns:", len(customer_final.columns))

===== FINAL SEGMENTATION SAVED =====
Path: ../data/processed/final_customer_segmentation.csv
Rows: 5350
Columns: 8


In [16]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


class CustomerFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, log_features):
        self.log_features = log_features

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        for col in self.log_features:
            X[col] = np.log1p(X[col])

        return X


# Production pipeline
final_pipeline = Pipeline([
    (
        "feature_transformer",
        CustomerFeatureTransformer(log_features=log_features)
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KMeans(
            n_clusters=final_k,
            random_state=final_random_state,
            n_init=final_n_init
        )
    )
])

# Fit the complete pipeline on all customer data
final_pipeline.fit(X)

print("===== PRODUCTION PIPELINE =====")
print(final_pipeline)

===== PRODUCTION PIPELINE =====
Pipeline(steps=[('feature_transformer',
                 CustomerFeatureTransformer(log_features=['Frequency',
                                                          'Monetary',
                                                          'Total_Quantity',
                                                          'Unique_Products',
                                                          'Average_Order_Value'])),
                ('scaler', StandardScaler()),
                ('model', KMeans(n_clusters=2, n_init=10, random_state=42))])


In [17]:
pipeline_labels = final_pipeline.predict(X)

same_predictions = np.array_equal(
    final_labels,
    pipeline_labels
)

print("===== PIPELINE VALIDATION =====")
print("Predictions match:", same_predictions)
print("Total predictions:", len(pipeline_labels))

print("\nPipeline cluster counts:")
print(pd.Series(pipeline_labels).value_counts().sort_index())

===== PIPELINE VALIDATION =====
Predictions match: True
Total predictions: 5350

Pipeline cluster counts:
0    2714
1    2636
Name: count, dtype: int64


In [18]:
import os
import joblib

model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(
    model_dir,
    "customer_segmentation_pipeline.joblib"
)

joblib.dump(
    final_pipeline,
    model_path
)

print("===== MODEL SAVED =====")
print("Path:", model_path)
print("File exists:", os.path.exists(model_path))
print("File size (MB):", round(os.path.getsize(model_path) / (1024 ** 2), 2))

===== MODEL SAVED =====
Path: ../models/customer_segmentation_pipeline.joblib
File exists: True
File size (MB): 0.02


In [19]:
loaded_pipeline = joblib.load(model_path)

loaded_labels = loaded_pipeline.predict(X)

print("===== LOADED MODEL VALIDATION =====")
print("Model loaded successfully:", loaded_pipeline is not None)
print("Predictions match original:", np.array_equal(final_labels, loaded_labels))

print("\nCluster counts:")
print(pd.Series(loaded_labels).value_counts().sort_index())

===== LOADED MODEL VALIDATION =====
Model loaded successfully: True
Predictions match original: True

Cluster counts:
0    2714
1    2636
Name: count, dtype: int64


In [20]:
import json

model_metadata = {
    "model": "KMeans",
    "n_clusters": final_k,
    "random_state": final_random_state,
    "n_init": final_n_init,
    "features": features,
    "log_features": log_features,
    "training_customers": len(X_final),
    "holdout_size": len(X_holdout),
    "holdout_metrics": {
        "silhouette": 0.3425,
        "davies_bouldin": 1.0832,
        "calinski_harabasz": 734.60
    },
    "final_metrics": {
        "silhouette": round(final_silhouette, 4),
        "davies_bouldin": round(final_davies_bouldin, 4),
        "calinski_harabasz": round(final_calinski_harabasz, 2)
    },
    "cluster_interpretation": {
        "0": "Inactive / Low-Value Customers",
        "1": "Active / High-Value Customers"
    }
}

metadata_path = "../models/customer_segmentation_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(model_metadata, f, indent=4)

print("===== MODEL METADATA SAVED =====")
print("Path:", metadata_path)
print("File exists:", os.path.exists(metadata_path))

===== MODEL METADATA SAVED =====
Path: ../models/customer_segmentation_metadata.json
File exists: True


In [21]:
with open(metadata_path, "r") as f:
    loaded_metadata = json.load(f)

print("===== METADATA VALIDATION =====")
print("Metadata loaded successfully:", loaded_metadata is not None)

print("\nModel:", loaded_metadata["model"])
print("Clusters:", loaded_metadata["n_clusters"])
print("Features:", loaded_metadata["features"])

print("\nHoldout metrics:")
print(loaded_metadata["holdout_metrics"])

print("\nFinal metrics:")
print(loaded_metadata["final_metrics"])

print("\nCluster interpretation:")
print(loaded_metadata["cluster_interpretation"])

===== METADATA VALIDATION =====
Metadata loaded successfully: True

Model: KMeans
Clusters: 2
Features: ['Recency', 'Frequency', 'Monetary', 'Total_Quantity', 'Unique_Products', 'Average_Order_Value']

Holdout metrics:
{'silhouette': 0.3425, 'davies_bouldin': 1.0832, 'calinski_harabasz': 734.6}

Final metrics:
{'silhouette': 0.3632, 'davies_bouldin': 1.026, 'calinski_harabasz': 4151.84}

Cluster interpretation:
{'0': 'Inactive / Low-Value Customers', '1': 'Active / High-Value Customers'}


In [22]:
final_summary = pd.DataFrame({
    "Metric": [
        "Algorithm",
        "Clusters",
        "Customers",
        "Holdout Customers",
        "Holdout Silhouette",
        "Holdout Davies-Bouldin",
        "Holdout Calinski-Harabasz",
        "Final Silhouette",
        "Final Davies-Bouldin",
        "Final Calinski-Harabasz"
    ],
    "Value": [
        "K-Means",
        final_k,
        len(X_final),
        len(X_holdout),
        0.3425,
        1.0832,
        734.60,
        final_silhouette,
        final_davies_bouldin,
        final_calinski_harabasz
    ]
})

print("===== FINAL MODEL SUMMARY =====")
display(final_summary)

===== FINAL MODEL SUMMARY =====


,Metric,Value
0,Algorithm,K-Means
1,Clusters,2
2,Customers,5350
3,Holdout Customers,1070
4,Holdout Silhouette,0.3425
5,Holdout Davies-Bouldin,1.0832
6,Holdout Calinski-Harabasz,734.6
7,Final Silhouette,0.36316
8,Final Davies-Bouldin,1.026002
9,Final Calinski-Harabasz,4151.837885
